# 02. Профилирование и очистка данных

**ЛР 2, шаг 7.** Цель: понять, насколько данные OSM о школах и поликлиниках Уфы пригодны для ответа на вопросы заказчика, найти проблемы и подготовить очищенные слои в \`data/processed/\`.

Проверяем по каждому слою: число объектов, CRS, типы геометрии, пустые и некорректные геометрии, дубликаты (по id и по положению), заполненность атрибутов, объекты за пределами территории.

Найденные проблемы оформлены задачами с меткой \`data\`, выводы перенесены в \`docs/data-quality.md\`.

[![Открыть в Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BariNiaz/pm-lab-2-schoolmed-access/blob/main/notebooks/02_data_profiling.ipynb)

## Настройка среды

Ноутбук работает в **Google Colab** (основной способ) и локально в Jupyter или VS Code. В Colab ячейка ниже ставит недостающие библиотеки и подключает Google Диск; локально работает с папками репозитория.

Перед первым запуском впишите в \`PROJECT_FOLDER\` имя папки группы на Google Диске. Запуск: **Среда выполнения → Выполнить все**.

In [ ]:
# === Настройка среды: запустите эту ячейку первой ===
import sys, subprocess
from pathlib import Path

# Имя папки группы в «Моём диске» Google. Обычно совпадает с именем репозитория.
PROJECT_FOLDER = "pm-lab-2-schoolmed-access"

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "geopandas>=1.0"], check=True)
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive") / PROJECT_FOLDER
    if not BASE.exists():
        raise FileNotFoundError(
            f"На Google Диске нет папки «{PROJECT_FOLDER}». Создайте её или добавьте ярлык "
            "общей папки группы в «Мой диск» (приложение Б, шаг 2) и запустите ячейку снова.")
else:
    BASE = Path("..")   # локальный запуск из папки notebooks/ репозитория

RAW, MANUAL, PROCESSED = BASE / "data" / "raw", BASE / "data" / "manual", BASE / "data" / "processed"
IMG, MAPS = BASE / "docs" / "img", BASE / "docs" / "maps"
for folder in (RAW, MANUAL, PROCESSED, IMG, MAPS):
    folder.mkdir(parents=True, exist_ok=True)
print("Среда:", "Google Colab" if IN_COLAB else "локальная", "· рабочая папка:", BASE.resolve())

In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt

LAYERS = ["districts", "schools", "clinics_osm"]   # слои из ноутбука 01
DUPLICATE_DISTANCE_M = 15          # объекты ближе этого расстояния считаем возможными дублями

boundary = gpd.read_file(RAW / "boundary.gpkg")
layers = {name: gpd.read_file(RAW / f"{name}.gpkg") for name in LAYERS}
METRIC_CRS = boundary.estimate_utm_crs()   # метрическая проекция UTM для расстояний и площадей
print("Метрическая система координат:", METRIC_CRS)
for name, gdf in layers.items():
    print(f"{name}: {len(gdf)} объектов, {gdf.geom_type.value_counts().to_dict()}")

## Сводный профиль слоёв

In [ ]:
def profile(name: str, gdf: gpd.GeoDataFrame) -> dict:
    g = gdf.geometry
    inside = gdf.intersects(boundary.to_crs(gdf.crs).union_all())
    dup_id = gdf.duplicated(subset=["osm_type", "osm_id"]).sum() if "osm_id" in gdf else None
    minx, miny, maxx, maxy = gdf.total_bounds
    return {
        "слой": name,
        "объектов": len(gdf),
        "CRS": gdf.crs.to_string() if gdf.crs else "НЕ ЗАДАНА",
        "типы геометрии": ", ".join(f"{k}: {v}" for k, v in g.geom_type.value_counts().items()),
        "пустых геометрий": int(g.is_empty.sum() + g.isna().sum()),
        "некорректных геометрий": int((~g.is_valid).sum()),
        "дубликатов по id": dup_id,
        "вне территории": int((~inside).sum()),
        "охват (lon/lat)": f"{minx:.3f}, {miny:.3f} – {maxx:.3f}, {maxy:.3f}",
    }

summary = pd.DataFrame([profile(n, g) for n, g in layers.items()])
summary.to_csv(PROCESSED / "profile_summary.csv", index=False)
print(summary.to_string(index=False))

## Заполненность атрибутов

Доля непустых значений по каждому атрибуту. Низкая заполненность важного атрибута — повод для задачи или для пересмотра требований. Для анализа доступности критичны координаты и название; адрес и режим работы нужны карте, но не расчёту.

In [ ]:
completeness = {}
for name, gdf in layers.items():
    attrs = gdf.drop(columns=["geometry", "osm_type", "osm_id"], errors="ignore")
    completeness[name] = (attrs.notna().mean() * 100).round(1)
completeness = pd.DataFrame(completeness).fillna("—")
print(completeness.to_string())

## Дубликаты по положению и состав тегов

Для точечных слоёв ищем объекты ближе \`DUPLICATE_DISTANCE_M\` метров друг к другу. Для поликлиник отдельно смотрим сочетания тегов \`amenity\` и \`healthcare\`: часть объектов — лаборатории, центры и частные кабинеты, а не поликлиники.

In [ ]:
def to_points(gdf):
    g = gdf.copy()
    g["geometry"] = g.geometry.representative_point()
    return g

def near_pairs(gdf, max_dist=DUPLICATE_DISTANCE_M):
    pts = to_points(gdf).to_crs(METRIC_CRS).reset_index(drop=True)
    geoms = list(pts.geometry)
    pairs = []
    for i in range(len(geoms)):
        for j in range(i + 1, len(geoms)):
            d = geoms[i].distance(geoms[j])
            if d <= max_dist:
                pairs.append((i, j, round(float(d), 1)))
    return pairs

schools_pairs = near_pairs(layers["schools"])
clinics_pairs = near_pairs(layers["clinics_osm"])
print("Пар школ ближе 15 м:", len(schools_pairs))
print("Пар поликлиник ближе 15 м:", len(clinics_pairs))
if len(clinics_pairs):
    print(clinics_pairs[:10])

print("\nСочетания тегов поликлиник:")
print(layers["clinics_osm"].groupby(
    [layers["clinics_osm"]["amenity"].fillna("—"), layers["clinics_osm"]["healthcare"].fillna("—")]
).size().rename("объектов").to_string())

print("\nУровни административных единиц в слое districts:")
print(layers["districts"]["admin_level"].value_counts().to_string())

## Очистка

Правила очистки — управленческие решения, они влияют на результат анализа. Каждое правило записано в \`docs/data-quality.md\`; спорные случаи (например, что считать поликлиникой) вынесены на согласование с заказчиком.

Правила:
1. **Районы:** оставляем только 7 городских районов Уфы (\`admin_level = 9\`); сельсоветы, районы области и регион отбрасываем.
2. **Поликлиники:** исключаем лаборатории, центры и частные кабинеты (\`healthcare = laboratory, centre, psychotherapist\`) — это не поликлиники.
3. **Точки:** полигоны школ и поликлиник заменяем на \`representative_point\`.
4. **Территория:** объекты за границей Уфы отбрасываем.
5. **Дубликаты:** из пар ближе 15 м оставляем один объект.

In [ ]:
CITY_DISTRICTS = [
    "Дёмский район", "Калининский район", "Кировский район", "Ленинский район",
    "Октябрьский район", "Орджоникидзевский район", "Советский район",
]
EXCLUDE_HEALTHCARE = ["laboratory", "centre", "psychotherapist"]

clean = {}
area = boundary.to_crs("EPSG:4326").union_all()

# 1. районы — только городские
d = layers["districts"].copy()
d["name"] = d["name"].astype(str).str.strip()
d = d[(d["admin_level"].astype(str) == "9") & d["name"].isin(CITY_DISTRICTS)]
d = d[~d.geometry.is_empty & d.geometry.notna()].copy()
clean["districts"] = d

# 2. поликлиники — только поликлиники
c = layers["clinics_osm"].copy()
c["healthcare"] = c["healthcare"].fillna("")
c = c[~c["healthcare"].isin(EXCLUDE_HEALTHCARE)]
c = c[c["amenity"].isin(["clinic", "doctors"]) | c["healthcare"].isin(["clinic", "doctor"])]

# 3-4. точки и только внутри территории
def prepare_points(gdf):
    g = gdf[gdf.geometry.notna() & ~gdf.geometry.is_empty].copy()
    g = g[g.intersects(area)]
    g = to_points(g)
    g = g.drop_duplicates(subset=["osm_type", "osm_id"])
    return g

s = prepare_points(layers["schools"])
c = prepare_points(c)

# 5. дубликаты по положению
def drop_close(gdf, max_dist=DUPLICATE_DISTANCE_M):
    gdf = gdf.to_crs(METRIC_CRS).reset_index(drop=True)
    keep = []
    for i, pt in enumerate(gdf.geometry):
        if all(pt.distance(gdf.geometry[j]) > max_dist for j in keep):
            keep.append(i)
    return gdf.loc[keep].to_crs("EPSG:4326").reset_index(drop=True)

s = drop_close(s)
c = drop_close(c)

clean["schools"] = s
clean["clinics"] = c
clean["boundary"] = boundary.to_crs("EPSG:4326")

print(pd.DataFrame({
    "было": {n: len(layers[n]) for n in LAYERS},
    "стало": {"districts": len(clean["districts"]), "schools": len(clean["schools"]), "clinics_osm": len(clean["clinics"])},
}).to_string())
print("\nРайоны:", sorted(clean["districts"]["name"].tolist()))
print("Школ:", len(clean["schools"]), "· поликлиник:", len(clean["clinics"]))

for name, g in clean.items():
    g.to_file(PROCESSED / f"{name}.gpkg", driver="GPKG")
print("\nСохранено в data/processed:", sorted(p.name for p in PROCESSED.glob("*.gpkg")))

## Карта для визуальной проверки

In [ ]:
fig, ax = plt.subplots(figsize=(9, 9))
clean["districts"].plot(ax=ax, color="whitesmoke", edgecolor="gray")
clean["schools"].plot(ax=ax, color="#1f6feb", markersize=8, label="Школы")
clean["clinics"].plot(ax=ax, color="#d1242f", markersize=8, label="Поликлиники")
ax.set_title("Очищенные данные: школы и поликлиники Уфы (© участники OpenStreetMap)")
ax.legend(loc="lower left")
ax.set_axis_off()
fig.savefig(IMG / "data_check.png", dpi=150, bbox_inches="tight")
print("Сохранено:", IMG / "data_check.png")

## Перенос результатов в репозиторий

Очищенные слои из \`data/processed\` попадают в репозиторий через Pull Request. В Colab ячейка ниже собирает архив \`results_02.zip\`: распакуйте его и загрузите файлы в те же папки репозитория в своей ветке (**Add file → Upload files**). При локальной работе файлы уже лежат в репозитории, ячейку можно пропустить.

In [ ]:
import shutil
FOLDERS = [PROCESSED, IMG]
if IN_COLAB:
    from google.colab import files
    pack = Path("/content/results_02")
    shutil.rmtree(pack, ignore_errors=True)
    for folder in FOLDERS:
        for f in folder.rglob("*"):
            if f.is_file() and f.name != ".gitkeep":
                if f.stat().st_size > 10 * 1024 * 1024:
                    print(f"Пропущен (больше 10 МБ): {f.relative_to(BASE)}")
                    continue
                target = pack / f.relative_to(BASE)
                target.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(f, target)
    archive = shutil.make_archive(str(pack), "zip", pack)
    print("В архиве:", sorted(str(p.relative_to(pack)) for p in pack.rglob("*") if p.is_file()))
    files.download(archive)
else:
    print("Локальный запуск: результаты уже лежат в папках репозитория")

## Выводы

Числа и выводы перенесены в \`docs/data-quality.md\`, проблемы оформлены задачами с меткой \`data\`:

1. В слое районов 31 объект разных административных уровней вместо 7 городских районов — правило очистки оставляет только \`admin_level = 9\`.
2. 71 объект в слое поликлиник — лаборатории, диагностические центры и частные кабинеты, а не поликлиники — исключены правилом по тегу \`healthcare\`.
3. Адреса, сайт и режим работы школ заполнены редко (адрес — 11 %, режим работы — 4 %) — принято как ограничение, на расчёт расстояний не влияет.
4. Четыре поликлиники находятся вне границы Уфы — отброшены правилом очистки.

Какие проблемы требуют решения заказчика: что считать поликлиникой (лаборатории и центры), нужно ли добирать адреса и режим работы школ вручную.